In [1]:
import pandas as pd
import sys
sys.path.append("../streamlit_app")
from utils import parse_latlng

import requests

url = "https://archive-api.open-meteo.com/v1/archive"

# ── 0. Charger les données brutes ──
df = pd.read_csv("../data/activities.csv")

# ── 1. Garder uniquement les courses à pied ──
df = df[df["sport_type"] == "Run"].copy()
print(f"Runs only: {len(df)} activities")

# ── 2. Convertir les dates ──
df["start_date_local"] = pd.to_datetime(df["start_date_local"], utc=True)

# ── 3. Ajouter des colonnes utiles ──
jours_fr = {
    "Monday": "Lundi", "Tuesday": "Mardi", "Wednesday": "Mercredi",
    "Thursday": "Jeudi", "Friday": "Vendredi",
    "Saturday": "Samedi", "Sunday": "Dimanche"
}
mois_fr = {
    1: "janvier", 2: "février", 3: "mars", 4: "avril",
    5: "mai", 6: "juin", 7: "juillet", 8: "août",
    9: "septembre", 10: "octobre", 11: "novembre", 12: "décembre"
}

df["day_of_week"] = df["start_date_local"].dt.day_name().map(jours_fr)
df["date_fr"]     = df["start_date_local"].apply(
    lambda x: f"{x.day} {mois_fr[x.month]} {x.year}"
)
df["hour_fr"]     = df["start_date_local"].apply(
    lambda x: f"{x.hour}h{x.minute:02d}"
)
df["date"]        = df["start_date_local"].dt.date
df["hour"]        = df["start_date_local"].dt.hour
df["month"]       = df["start_date_local"].dt.tz_localize(None).dt.to_period("M")
df["year"]        = df["start_date_local"].dt.year

# ── 4. Convertir les unités ──
df["distance_km"]   = df["distance_m"] / 1000
df["pace_min_km"]   = (1 / df["average_speed_m_s"]) / 60 * 1000
df["pause_seconds"] = df["elapsed_time_seconds"] - df["moving_time_seconds"]

# ── 5. Sauvegarder df_map avant de dropper (pour la carte) ──
df_map = df[["id", "map_id", "summary_polyline",
             "start_latlng", "end_latlng",
             "location_city", "date"]].copy()

# ── 5.5 Parser les coordonnées (temporaire pour météo) ──
df["coords"] = df["start_latlng"].apply(parse_latlng)
df["lat"] = df["coords"].apply(lambda x: x[0] if x else None)
df["lon"] = df["coords"].apply(lambda x: x[1] if x else None)

# ── 5.6 Enrichissement météo avec logique delta ──
WEATHER_COLS = ["weather_temperature", "weather_humidity",
                "weather_precipitation", "weather_code"]

# Initialiser les colonnes météo à None
for col in WEATHER_COLS:
    df[col] = None

# Charger les données météo existantes si disponibles
clean_path = "../data/activities_clean.csv"
try:
    df_existing = pd.read_csv(clean_path)
    existing_weather_cols = [c for c in WEATHER_COLS if c in df_existing.columns]
    if existing_weather_cols:
        df_existing = df_existing[["id"] + existing_weather_cols]
        df = df.drop(columns=existing_weather_cols, errors="ignore")
        df = df.merge(df_existing, on="id", how="left")
        print(f"Météo existante chargée pour {df['weather_temperature'].notna().sum()} sorties")
except FileNotFoundError:
    print("Pas de fichier existant, fetch complet")

# Fetch météo uniquement pour les sorties sans données
fetched = 0
for idx, row in df.iterrows():
    if pd.notna(row.get("weather_temperature")):
        continue

    if pd.isna(row["lat"]) or pd.isna(row["lon"]):
        continue

    if fetched % 10 == 0:
        print(f"  Fetching activity {fetched}...")

    params = {
        "latitude": row["lat"],
        "longitude": row["lon"],
        "start_date": str(row["date"]),
        "end_date": str(row["date"]),
        "hourly": ["temperature_2m", "relative_humidity_2m",
                   "precipitation", "weather_code"],
        "timezone": "Europe/Paris"
    }
    try:
        response = requests.get(url, params=params).json()
        hour_idx = int(row["hour"])
        df.at[idx, "weather_temperature"]   = response["hourly"]["temperature_2m"][hour_idx]
        df.at[idx, "weather_humidity"]       = response["hourly"]["relative_humidity_2m"][hour_idx]
        df.at[idx, "weather_precipitation"]  = response["hourly"]["precipitation"][hour_idx]
        df.at[idx, "weather_code"]           = response["hourly"]["weather_code"][hour_idx]
        fetched += 1
    except Exception as e:
        print(f"Erreur pour idx {idx} : {e}")

print(f"Météo fetchée pour {fetched} nouvelles sorties")

# ── 6. Supprimer les colonnes inutiles pour l'analyse ──
cols_to_drop = [
    "external_id", "upload_id", "type", "start_date",
    "timezone", "workout_type", "private", "visibility",
    "commute", "trainer", "utc_offset", "has_heartrate",
    "average_cadence",
    "start_latlng", "end_latlng", "map_id",
    "summary_polyline", "gear_id",
    "location_city", "location_state", "location_country",
    "distance_m", "elapsed_time_seconds",
    "average_watts", "max_watts", "weighted_average_watts",
    "device_watts", "kilojoules",
    "kudos_count", "comment_count", "photo_count",
    "total_photo_count", "achievement_count",
    "coords", "lat", "lon",
]
df = df.drop(columns=cols_to_drop)

print(f"Colonnes restantes : {list(df.columns)}")
print(f"Shape : {df.shape}")

# ── 7. Sauvegarder ──
df.to_csv("../data/activities_clean.csv", index=False)
df_map.to_csv("../data/activities_map.csv", index=False)

print("Activities saved to data/activities_clean.csv and data/activities_map.csv")


2026-08-13 21:29:08.464 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


Runs only: 119 activities
Météo existante chargée pour 118 sorties
  Fetching activity 0...
Météo fetchée pour 1 nouvelles sorties
Colonnes restantes : ['id', 'name', 'sport_type', 'manual', 'flagged', 'start_date_local', 'moving_time_seconds', 'average_speed_m_s', 'max_speed_m_s', 'total_elevation_gain_m', 'elev_high_m', 'elev_low_m', 'average_heartrate', 'max_heartrate', 'pr_count', 'suffer_score', 'day_of_week', 'date_fr', 'hour_fr', 'date', 'hour', 'month', 'year', 'distance_km', 'pace_min_km', 'pause_seconds', 'weather_temperature', 'weather_humidity', 'weather_precipitation', 'weather_code']
Shape : (119, 30)
Activities saved to data/activities_clean.csv and data/activities_map.csv
